# 01 — Data Exploration

**ECE1513 Course Project — Traffic Congestion Prediction near U of T St. George Campus**

This notebook performs an initial exploration of the raw traffic and weather datasets. The goals are to:

1. Load the City of Toronto midblock speed-bin data and filter to locations within ~1.5 km of U of T St. George campus.
2. Understand the structure of the speed-bin columns and compute average speeds.
3. Map the locations (lat/lon scatter plot).
4. Visualize speed patterns by hour of day and day of week.
5. Load and preview the weather data.
6. Summarize data quality findings.

In [ ]:
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100

%matplotlib inline

## 1. Load Raw Traffic Data

The traffic data comes from the City of Toronto **"Traffic Volumes — Midblock Vehicle Speed, Volume and Classification Counts"** dataset. The speed file contains vehicle counts in speed bins (1-19 kph, 20-25 kph, ..., 81-160 kph) for each location, direction, and 15-minute time interval.

In [ ]:
traffic_raw = pd.read_csv('../data/raw/svc_raw_data_speed_2020_2024.csv')
print(f'Raw traffic data shape: {traffic_raw.shape}')
print(f'Columns: {list(traffic_raw.columns)}')
traffic_raw.head()

In [ ]:
print('--- Data Types ---')
print(traffic_raw.dtypes)
print()
print('--- Basic Stats ---')
traffic_raw.describe()

## 2. Filter to U of T St. George Area

We restrict our analysis to locations within approximately 1.5 km of the U of T St. George campus. The bounding box is:
- Latitude: 43.645 to 43.675
- Longitude: -79.415 to -79.385

In [ ]:
LAT_MIN, LAT_MAX = 43.645, 43.675
LON_MIN, LON_MAX = -79.415, -79.385

mask = (
    (traffic_raw['latitude'] >= LAT_MIN) & (traffic_raw['latitude'] <= LAT_MAX) &
    (traffic_raw['longitude'] >= LON_MIN) & (traffic_raw['longitude'] <= LON_MAX)
)
traffic_df = traffic_raw[mask].copy()

print(f'Filtered to U of T area: {traffic_df.shape[0]:,} rows '
      f'(from {traffic_raw.shape[0]:,} total, {mask.sum()/len(traffic_raw)*100:.1f}%)')
print(f'Unique locations: {traffic_df["location_name"].nunique()}')

In [ ]:
# Show all unique locations
locations = traffic_df.groupby('location_name').agg(
    lat=('latitude', 'first'),
    lon=('longitude', 'first'),
    records=('id', 'count')
).sort_values('records', ascending=False)

print(f'Number of unique locations: {len(locations)}')
locations.head(20)

## 3. Map of Locations

A scatter plot of all sensor locations within our bounding box, giving a sense of spatial coverage near the U of T campus.

In [ ]:
loc_coords = traffic_df.drop_duplicates(subset='location_name')[['location_name', 'latitude', 'longitude']]

fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(loc_coords['longitude'], loc_coords['latitude'],
           s=40, alpha=0.7, edgecolors='black', linewidth=0.5, c='steelblue')

# Mark approximate U of T campus centre
ax.scatter(-79.3957, 43.6629, s=200, marker='*', c='red', zorder=5, label='U of T (approx)')

ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title(f'Traffic Sensor Locations near U of T St. George ({len(loc_coords)} locations)')
ax.legend()

# Draw bounding box
from matplotlib.patches import Rectangle
rect = Rectangle((LON_MIN, LAT_MIN), LON_MAX - LON_MIN, LAT_MAX - LAT_MIN,
                  linewidth=1.5, edgecolor='gray', facecolor='none', linestyle='--')
ax.add_patch(rect)

plt.tight_layout()
os.makedirs('../results/figures', exist_ok=True)
plt.savefig('../results/figures/location_map.png', bbox_inches='tight')
plt.show()

## 4. Speed Bin Distribution

The data does not provide a single speed value per record. Instead, it provides vehicle **counts** in each speed bin. We can examine the distribution of counts across bins and compute a weighted average speed.

In [ ]:
# Speed bin columns and their midpoints (km/h)
speed_bin_cols = [
    'vol_1_19kph', 'vol_20_25kph', 'vol_26_30kph', 'vol_31_35kph',
    'vol_36_40kph', 'vol_41_45kph', 'vol_46_50kph', 'vol_51_55kph',
    'vol_56_60kph', 'vol_61_65kph', 'vol_66_70kph', 'vol_71_75kph',
    'vol_76_80kph', 'vol_81_160kph'
]

speed_bin_midpoints = np.array([10.0, 22.5, 28.0, 33.0, 38.0, 43.0, 48.0,
                                53.0, 58.0, 63.0, 68.0, 73.0, 78.0, 120.5])

# Total volume per bin across all records
bin_totals = traffic_df[speed_bin_cols].sum()

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(range(len(speed_bin_cols)), bin_totals.values, color='steelblue', edgecolor='white')
ax.set_xticks(range(len(speed_bin_cols)))
ax.set_xticklabels([c.replace('vol_', '').replace('kph', '') for c in speed_bin_cols], rotation=45, ha='right')
ax.set_xlabel('Speed Bin (km/h)')
ax.set_ylabel('Total Vehicle Count')
ax.set_title('Aggregate Vehicle Counts by Speed Bin (U of T Area, 2020-2024)')
plt.tight_layout()
plt.savefig('../results/figures/speed_bin_distribution.png', bbox_inches='tight')
plt.show()

## 5. Compute Average Speed

We compute the weighted average speed for each record using the bin midpoints weighted by the vehicle counts in each bin.

In [ ]:
bin_values = traffic_df[speed_bin_cols].values
total_vehicles = bin_values.sum(axis=1)

# Avoid division by zero for records with no vehicles
with np.errstate(divide='ignore', invalid='ignore'):
    avg_speed = (bin_values * speed_bin_midpoints).sum(axis=1) / total_vehicles
    avg_speed = np.where(total_vehicles > 0, avg_speed, np.nan)

traffic_df['avg_speed'] = avg_speed
traffic_df['total_volume'] = total_vehicles

print(f'Average speed stats:')
print(traffic_df['avg_speed'].describe())
print(f'\nRecords with zero vehicles: {(total_vehicles == 0).sum():,}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(traffic_df['avg_speed'].dropna(), bins=60, edgecolor='white', color='steelblue')
axes[0].set_xlabel('Average Speed (km/h)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Average Speed (U of T Area)')
axes[0].axvline(traffic_df['avg_speed'].median(), color='red', linestyle='--',
                label=f'Median: {traffic_df["avg_speed"].median():.1f} km/h')
axes[0].legend()

traffic_df.boxplot(column='avg_speed', ax=axes[1])
axes[1].set_title('Box Plot of Average Speed')
axes[1].set_ylabel('Speed (km/h)')

plt.tight_layout()
plt.savefig('../results/figures/speed_distribution.png', bbox_inches='tight')
plt.show()

## 6. Speed Patterns by Hour of Day and Day of Week

These temporal patterns are key signals for congestion prediction on city streets.

In [ ]:
traffic_df['time_start'] = pd.to_datetime(traffic_df['time_start'])
traffic_df['hour'] = traffic_df['time_start'].dt.hour
traffic_df['day_of_week'] = traffic_df['time_start'].dt.day_name()
traffic_df['day_of_week_num'] = traffic_df['time_start'].dt.dayofweek

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Speed by hour
hourly = traffic_df.groupby('hour')['avg_speed'].agg(['mean', 'std']).reset_index()
axes[0].bar(hourly['hour'], hourly['mean'], yerr=hourly['std'], color='steelblue',
            edgecolor='white', capsize=2, alpha=0.8)
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Mean Speed (km/h)')
axes[0].set_title('Average Speed by Hour of Day')
axes[0].set_xticks(range(24))

# Speed by day of week
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
daily = traffic_df.groupby('day_of_week')['avg_speed'].mean().reindex(day_order)
daily.plot(kind='bar', ax=axes[1], color='darkorange', edgecolor='white')
axes[1].set_xlabel('Day of Week')
axes[1].set_ylabel('Mean Speed (km/h)')
axes[1].set_title('Average Speed by Day of Week')

plt.tight_layout()
plt.savefig('../results/figures/speed_by_hour_day.png', bbox_inches='tight')
plt.show()

In [ ]:
# Heatmap: hour x day_of_week
day_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
heatmap_data = traffic_df.pivot_table(
    values='avg_speed', index='hour', columns='day_of_week_num', aggfunc='mean'
)
heatmap_data.columns = [day_labels[i] for i in heatmap_data.columns]

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(heatmap_data, annot=True, fmt='.0f', cmap='RdYlGn', linewidths=0.5,
            cbar_kws={'label': 'Avg Speed (km/h)'}, ax=ax)
ax.set_title('Average Speed by Hour and Day of Week')
ax.set_ylabel('Hour of Day')
ax.set_xlabel('Day of Week')
plt.tight_layout()
plt.savefig('../results/figures/speed_heatmap_exploration.png', bbox_inches='tight')
plt.show()

## 7. Load and Preview Weather Data

Weather data comes from Environment Canada (Toronto Pearson station), downloaded via the GeoMet API. Monthly CSVs are stored in `data/raw/weather/`.

In [ ]:
weather_files = sorted(glob.glob('../data/raw/weather/toronto_pearson_*.csv'))
print(f'Found {len(weather_files)} weather CSV files')

weather_dfs = []
for f in weather_files:
    df = pd.read_csv(f)
    weather_dfs.append(df)

weather_df = pd.concat(weather_dfs, ignore_index=True)
print(f'Combined weather data shape: {weather_df.shape}')
print(f'Columns: {list(weather_df.columns)}')
weather_df.head()

In [ ]:
print('--- Weather Data Types ---')
print(weather_df.dtypes)
print()
print('--- Key Weather Columns ---')
key_weather_cols = ['TEMP', 'DEW_POINT_TEMP', 'RELATIVE_HUMIDITY', 'PRECIP_AMOUNT',
                    'WIND_SPEED', 'VISIBILITY', 'WEATHER_ENG_DESC']
for col in key_weather_cols:
    if col in weather_df.columns:
        non_null = weather_df[col].notna().sum()
        pct = non_null / len(weather_df) * 100
        print(f'  {col:25s}: {non_null:>6,} non-null ({pct:.1f}%)')

In [ ]:
# Missing values in weather data
missing_w = weather_df.isnull().sum()
missing_w_pct = (missing_w / len(weather_df) * 100).round(2)
missing_summary = pd.DataFrame({'missing_count': missing_w, 'missing_pct': missing_w_pct})
missing_summary = missing_summary.query('missing_count > 0').sort_values('missing_pct', ascending=False)
print(f'Columns with missing values: {len(missing_summary)}')
missing_summary

## 8. Data Quality Summary — Traffic Data

In [ ]:
# Missing values in traffic data
missing_t = traffic_df.isnull().sum()
missing_t_pct = (missing_t / len(traffic_df) * 100).round(2)
missing_t_summary = pd.DataFrame({'missing_count': missing_t, 'missing_pct': missing_t_pct})
missing_t_summary = missing_t_summary.query('missing_count > 0').sort_values('missing_pct', ascending=False)
print('Traffic data — columns with missing values:')
missing_t_summary

In [ ]:
# Time coverage
print(f'Time range: {traffic_df["time_start"].min()} to {traffic_df["time_start"].max()}')
print(f'Unique locations: {traffic_df["location_name"].nunique()}')
print(f'Unique directions: {traffic_df["direction"].unique()}')
print(f'Total records (U of T area): {len(traffic_df):,}')
print(f'Records with avg_speed NaN: {traffic_df["avg_speed"].isna().sum():,}')
print(f'Records with zero total volume: {(traffic_df["total_volume"] == 0).sum():,}')

## 9. Summary of Findings

Key observations from the exploratory analysis:

- **Dataset scope**: The speed-bin dataset covers 2020-2024. After filtering to the U of T St. George area (lat 43.645-43.675, lon -79.415 to -79.385), we retain ~104 unique sensor locations with substantial record counts.
- **Speed bins**: The data provides vehicle counts in 14 speed bins rather than a single speed value. We compute a weighted average speed from bin midpoints. For city streets, most vehicles fall in the 20-50 km/h range.
- **Speed distribution**: The average speed distribution reflects urban driving conditions, with a peak around 30-45 km/h — much lower than highway data.
- **Temporal patterns**: Clear speed dips during morning (7-9 AM) and evening (4-7 PM) rush hours on weekdays. Weekend speeds are noticeably higher.
- **Weather data**: 60 monthly CSVs from Toronto Pearson provide hourly temperature, humidity, precipitation, wind, and visibility. Some columns (e.g., PRECIP_AMOUNT, HUMIDEX) have significant missingness that will need handling.
- **Data quality**: Some records have zero total volume (no vehicles counted in any bin), which will produce NaN average speeds. These will be removed during preprocessing.

These insights will guide the preprocessing and feature engineering steps in the next notebook.